In [1]:
#!/usr/bin/env python3
"""
Raccolta dati OSM per l'indice di attrattività comunale.

Questo script scarica:
1. I confini comunali della Sardegna dal repository openpolis/geojson-italy (dati ISTAT)
2. I POI (Points of Interest) da OpenStreetMap tramite Overpass API

I POI vengono categorizzati in 5 pilastri tematici e associati ai comuni
tramite spatial join con geopandas.

Fonte confini: openpolis/geojson-italy (ISTAT)
Fonte POI: Overpass API (OpenStreetMap)

Dipendenze: pip install requests geopandas shapely pandas

Uso:
    python 01_raccolta_dati_osm.py

Output:
    - comuni_sardegna.geojson: confini comunali filtrati
    - osm_per_comune.csv: conteggi POI per comune e pilastro
"""


"\nRaccolta dati OSM per l'indice di attrattività comunale.\n\nQuesto script scarica:\n1. I confini comunali della Sardegna dal repository openpolis/geojson-italy (dati ISTAT)\n2. I POI (Points of Interest) da OpenStreetMap tramite Overpass API\n\nI POI vengono categorizzati in 5 pilastri tematici e associati ai comuni\ntramite spatial join con geopandas.\n\nFonte confini: openpolis/geojson-italy (ISTAT)\nFonte POI: Overpass API (OpenStreetMap)\n\nDipendenze: pip install requests geopandas shapely pandas\n\nUso:\n    python 01_raccolta_dati_osm.py\n\nOutput:\n    - comuni_sardegna.geojson: confini comunali filtrati\n    - osm_per_comune.csv: conteggi POI per comune e pilastro\n"

In [3]:
import json
import time
import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from pathlib import Path

In [4]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

# Regione target (modificare per scalare ad altre regioni)
# Il nome deve corrispondere alla colonna 'reg_name' nel GeoJSON ISTAT
REGION_NAME = "Sardegna"

# Directory di output per i file generati
OUTPUT_DIR = "/home/davide/Scaricati/sardegna-overtourism-aida26-main/output_attrattivita"

# URL del GeoJSON nazionale con tutti i comuni italiani (fonte: ISTAT via openpolis)
GEOJSON_URL = "https://raw.githubusercontent.com/openpolis/geojson-italy/master/geojson/limits_IT_municipalities.geojson"


In [5]:
import os
os.getcwd()

'/home/davide/codice/sardegna-overtourism-aida26/notebooks'

In [6]:
# Endpoint Overpass API, in ordine di tentativo. Il primo è l'istanza
# ufficiale; gli altri sono mirror pubblici usati come fallback se il
# primo restituisce errore/timeout (capita spesso con query pesanti).
# NOTA: se esegui questo script da un ambiente con proxy/whitelist di rete
# (come una sandbox), assicurati che questi host siano raggiungibili;
# altrimenti aggiungi/sostituisci con un endpoint permesso dal tuo proxy.
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

In [7]:
# Bounding box della Sardegna (sud, ovest, nord, est) in gradi decimali
# NOTA: Questa bbox viene usata come fallback; lo script tenta di calcolarla
# automaticamente dal GeoJSON filtrato per maggiore precisione
BBOX_SARDENYA = (38.8, 8.1, 41.4, 9.9)


In [8]:
# Timeout per le richieste HTTP (secondi)
REQUEST_TIMEOUT = 120


In [9]:
# ============================================================
# DEFINIZIONE PILASTRI OSM
# ============================================================
# Ogni pilastro raggruppa tag OSM correlati per tematica.
# I tag sono espressi come dizionari {"key": ..., "value": ...}.
# Se value è None, matcha qualsiasi valore per quella chiave.

PILLARS = {
    # -------- TURISMO --------
    # Hotel, ostelli, musei, attrazioni, spiagge, monumenti storici
    "turismo": [
        {"key": "tourism", "value": "hotel"},
        {"key": "tourism", "value": "hostel"},
        {"key": "tourism", "value": "guest_house"},
        {"key": "tourism", "value": "camp_site"},
        {"key": "tourism", "value": "caravan_site"},
        {"key": "tourism", "value": "motel"},
        {"key": "tourism", "value": "chalet"},
        {"key": "tourism", "value": "apartment"},
        {"key": "tourism", "value": "wilderness_hut"},
        {"key": "tourism", "value": "alpine_hut"},
        {"key": "tourism", "value": "museum"},
        {"key": "tourism", "value": "attraction"},
        {"key": "tourism", "value": "viewpoint"},
        {"key": "tourism", "value": "gallery"},
        {"key": "tourism", "value": "theme_park"},
        {"key": "tourism", "value": "zoo"},
        {"key": "tourism", "value": "picnic_site"},
        {"key": "tourism", "value": "artwork"},
        {"key": "natural", "value": "beach"},
        {"key": "historic", "value": None},  # Qualsiasi valore
    ],

    # -------- NATURA --------
    # Riserve naturali, parchi, aree protette, formazioni naturali
    "natura": [
        {"key": "leisure", "value": "nature_reserve"},
        {"key": "leisure", "value": "park"},
        {"key": "leisure", "value": "marina"},
        {"key": "boundary", "value": "protected_area"},
        {"key": "natural", "value": "wood"},
        {"key": "natural", "value": "water"},
        {"key": "natural", "value": "peak"},
        {"key": "natural", "value": "arch"},
        {"key": "natural", "value": "spring"},
        {"key": "natural", "value": "tree"},
        {"key": "natural", "value": "bay"},
        {"key": "natural", "value": "cape"},
        {"key": "natural", "value": "cliff"},
        {"key": "natural", "value": "rock"},
        {"key": "natural", "value": "rocks"},
        {"key": "natural", "value": "saddle"},
        {"key": "natural", "value": "gorge"},
        {"key": "natural", "value": "cave_entrance"},
        {"key": "natural", "value": "hot_spring"},
    ],

    # -------- SERVIZI --------
    # Ospedali, farmacie, scuole, università, banche, uffici postali
    "servizi": [
        {"key": "amenity", "value": "hospital"},
        {"key": "amenity", "value": "clinic"},
        {"key": "amenity", "value": "pharmacy"},
        {"key": "amenity", "value": "school"},
        {"key": "amenity", "value": "university"},
        {"key": "amenity", "value": "bank"},
        {"key": "amenity", "value": "post_office"},
    ],

    # -------- RISTORAZIONE --------
    # Ristoranti, bar, caffè, pub, gelaterie, negozi
    "ristorazione": [
        {"key": "amenity", "value": "restaurant"},
        {"key": "amenity", "value": "bar"},
        {"key": "amenity", "value": "cafe"},
        {"key": "amenity", "value": "pub"},
        {"key": "amenity", "value": "ice_cream"},
        {"key": "amenity", "value": "fast_food"},
        {"key": "amenity", "value": "biergarten"},
        {"key": "amenity", "value": "food_court"},
        #{"key": "shop", "value": None},  per Qualsiasi negozio - esclusa: non ha molto senso e fa impazzire la request
        {"key": "shop", "value": "pastry"},
        {"key": "shop", "value": "deli"},
        {"key": "shop", "value": "coffee"},
        {"key": "shop", "value": "tea"},
        {"key": "shop", "value": "confectionery"},
    ],

    # -------- INFRASTRUTTURE --------
    # Fermate bus, stazioni trasporto pubblico, porti, aeroporti
    "infrastrutture": [
        {"key": "public_transport", "value": "stop_position"},
        {"key": "public_transport", "value": "platform"},
        {"key": "public_transport", "value": "station"},
        {"key": "highway", "value": "bus_stop"},
        {"key": "aeroway", "value": "aerodrome"},
        {"key": "aeroway", "value": "helipad"},
        {"key": "aeroway", "value": "airstrip"},
        {"key": "harbour", "value": None},
        {"key": "amenity", "value": "ferry_terminal"},
    ],
}

In [10]:
# ============================================================
# FUNZIONI DI UTILITÀ
# ============================================================

def download_geojson(url: str, timeout: int = 60) -> dict:
    """Scarica un file GeoJSON da URL."""
    print(f"[1/6] Scaricamento confini comunali da {url}...")
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


In [11]:
def filter_region(geojson_data: dict, region_name: str) -> dict:
    """Filtra il GeoJSON per regione, mantenendo solo i comuni della regione target."""
    print(f"[2/6] Filtraggio per regione: {region_name}...")

    features = geojson_data.get("features", [])
    filtered = [
        f for f in features
        if f.get("properties", {}).get("reg_name") == region_name
    ]

    print(f"  → {len(filtered)} comuni trovati in {region_name}")

    result = {
        "type": "FeatureCollection",
        "features": filtered
    }
    return result

In [12]:
def calculate_bbox(geojson_data: dict) -> tuple:
    """Calcola la bounding box automaticamente dalla geometria filtrata."""
    from shapely.geometry import shape

    all_coords = []
    for feature in geojson_data.get("features", []):
        geom = shape(feature["geometry"])
        bounds = geom.bounds  # (minx, miny, maxx, maxy)
        all_coords.append(bounds)

    if not all_coords:
        raise ValueError("Nessuna geometria trovata nel GeoJSON")

    minx = min(c[0] for c in all_coords)
    miny = min(c[1] for c in all_coords)
    maxx = max(c[2] for c in all_coords)
    maxy = max(c[3] for c in all_coords)

    # Formato: (sud, ovest, nord, est)
    bbox = (miny, minx, maxy, maxx)
    print(f"  → Bounding box calcolata: {bbox}")
    return bbox

In [13]:
def _tag_selector(tag: dict) -> str:
    """
    Costruisce il selettore Overpass QL per un singolo tag.

    Se value è None, matcha qualsiasi valore per quella chiave
    (semplice filtro di esistenza `["key"]`).
    """
    key = tag["key"]
    value = tag["value"]
    if value is None:
        return f'["{key}"]'
    return f'["{key}"="{value}"]'

In [14]:
def build_overpass_query(bbox: tuple, pillar_tags: list) -> str:
    """
    Costruisce una query Overpass per ottenere nodi e way con tag specifici.

    IMPORTANTE: in Overpass QL non esiste un OR "dentro" un singolo
    selettore concatenando le condizioni con ";" (quello è un separatore
    di statement, non una disgiunzione logica). L'OR tra più tag si
    ottiene unendo più statement separati dentro lo stesso blocco `( ... )`,
    che è quello che facciamo qui: uno statement node/way per ogni tag
    del pilastro.

    Usiamo inoltre `out center;` invece di `out body; >; out skel qt;`:
    per i way questo restituisce direttamente il centroide (`center`),
    evitando di dover scaricare e ricostruire a mano la geometria dei
    nodi che compongono il way.
    """
    south, west, north, east = bbox

    statements = []
    for tag in pillar_tags:
        selector = _tag_selector(tag)
        statements.append(f'  node{selector}({south},{west},{north},{east});')
        statements.append(f'  way{selector}({south},{west},{north},{east});')

    body = "\n".join(statements)

    query = f"""[out:json][timeout:300];
(
{body}
);
out center;"""
    return query

In [15]:
# Header User-Agent descrittivo, come richiesto dalla policy d'uso di OSM/Overpass
# (le richieste con uno User-Agent generico tipo "python-requests/x.x" vengono
# spesso rifiutate con 406 dai mirror pubblici, incluso quello ufficiale).
# NOTA: personalizza il contatto con la tua email/repo, è buona pratica verso
# l'infrastruttura pubblica di OSM.
OVERPASS_HEADERS = {
    "User-Agent": "IndiceAttrattivitaComuni/1.0 (Master AIDA Bicocca; contatto: d.colucci4@campus.unibicocca.it)"
}

In [16]:
def query_overpass(
    endpoints: list,
    query: str,
    timeout: int = 300,
    max_retries_on_rate_limit: int = 1,
) -> dict:
    """
    Esegue una query sull'Overpass API, tentando in sequenza gli endpoint
    forniti finché uno non risponde con successo.

    In caso di 429 (Too Many Requests) attende il tempo indicato
    dall'header 'Retry-After' (o un default prudente) e ritenta lo
    STESSO endpoint prima di passare al successivo, invece di saltare
    subito al mirror dopo: bombardare più mirror in rapida successione
    peggiora la situazione e rischia di far scattare il rate limit anche
    sugli altri.
    """
    last_error = None
    for url in endpoints:
        attempt = 0
        while attempt <= max_retries_on_rate_limit:
            print(f"  → Esecuzione query Overpass su {url} (tentativo {attempt + 1})...")
            try:
                response = requests.post(
                    url, data={"data": query}, headers=OVERPASS_HEADERS, timeout=timeout
                )
                if response.status_code == 429:
                    retry_after = int(response.headers.get("Retry-After", 30))
                    print(f"    Rate limited (429). Attendo {retry_after}s prima di ritentare...")
                    time.sleep(retry_after)
                    attempt += 1
                    last_error = "429 Too Many Requests"
                    continue
                response.raise_for_status()
                return response.json()
            except requests.exceptions.HTTPError as e:
                print(f"    endpoint fallito ({url}): {e}")
                last_error = e
                break  # errore non di rate-limit: non ha senso ritentare lo stesso endpoint
            except Exception as e:
                print(f"    endpoint fallito ({url}): {e}")
                last_error = e
                break
        time.sleep(5)  # breve pausa di cortesia prima di provare il mirror successivo

    raise RuntimeError(f"Tutti gli endpoint Overpass hanno fallito: {last_error}")

In [17]:
def extract_points_from_overpass(data: dict) -> list:
    """
    Estrae i punti (nodi e centroidi dei way) dai risultati Overpass.

    Grazie a `out center;` nella query, i way includono un campo
    "center": {"lat":..., "lon":...} già calcolato da Overpass, quindi
    non serve ricostruire la geometria manualmente.
    """
    elements = data.get("elements", [])
    points = []

    for elem in elements:
        lat = None
        lon = None

        if elem.get("type") == "node":
            lat = elem.get("lat")
            lon = elem.get("lon")
        elif elem.get("type") == "way":
            center = elem.get("center", {})
            lat = center.get("lat")
            lon = center.get("lon")

        if lat is None or lon is None:
            continue

        points.append({
            "lat": lat,
            "lon": lon,
            "tags": elem.get("tags", {}),
            "osm_id": elem["id"],
            "osm_type": elem["type"],
        })

    return points

In [18]:
def classify_point(point: dict) -> list:
    """Classifica un punto POI in uno o più pilastri basandosi sui tag OSM."""
    tags = point.get("tags", {})
    matched_pillars = []

    for pillar_name, pillar_tags in PILLARS.items():
        for tag in pillar_tags:
            key = tag["key"]
            value = tag["value"]

            if key in tags:
                if value is None or tags[key] == value:
                    matched_pillars.append(pillar_name)
                    break  # Non serve controllare altri tag dello stesso pilastro

    return matched_pillars

In [19]:
def spatial_join_points_to_municipalities(
    points: list,
    comuni_gdf: gpd.GeoDataFrame
) -> gpd.GeoDataFrame:
    """Esegue lo spatial join tra punti POI e comuni per associare ogni POI al suo comune."""
    print("[4/6] Costruzione GeoDataFrame dei punti POI...")

    pts_data = [p for p in points if p.get("lat") is not None and p.get("lon") is not None]

    if not pts_data:
        print("  → Nessun punto valido trovato")
        return gpd.GeoDataFrame(columns=["lat", "lon", "osm_id", "osm_type", "tags"])

    geometry = [Point(p["lon"], p["lat"]) for p in pts_data]
    pts_gdf = gpd.GeoDataFrame(pts_data, geometry=geometry, crs="EPSG:4326")

    print(f"  → {len(pts_gdf)} punti creati")

    # Riproietta i comuni in un CRS proiettato per lo spatial join
    comuni_reprojected = comuni_gdf.to_crs(epsg=32632)  # UTM zone 32N
    pts_reprojected = pts_gdf.to_crs(epsg=32632)

    print("[5/6] Esecuzione spatial join...")
    joined = gpd.sjoin(
        pts_reprojected,
        comuni_reprojected,
        how="left",
        predicate="within"
    )

    print(f"  → {len(joined)} punti associati a comuni")
    return joined

In [20]:
def aggregate_counts(joined_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Aggrega i conteggi POI per comune e pilastro."""
    print("[6/6] Aggregazione conteggi per comune...")

    joined_gdf = joined_gdf.copy()
    joined_gdf["pillars"] = joined_gdf.apply(
        lambda row: classify_point({"tags": row.get("tags", {})}),
        axis=1
    )

    rows = []
    for _, row in joined_gdf.iterrows():
        pillars = row.get("pillars", [])
        if isinstance(pillars, list) and len(pillars) > 0:
            for pillar in pillars:
                rows.append({
                    "codice_istat": row.get("com_istat_code"),
                    "nome_comune": row.get("name"),
                    "pilastro": pillar
                })

    if not rows:
        print("  → Nessun punto classificato")
        return pd.DataFrame(columns=["codice_istat", "nome_comune", "pilastro", "count"])

    counts_df = pd.DataFrame(rows)

    agg_df = counts_df.groupby(["codice_istat", "nome_comune", "pilastro"]).size().reset_index(name="count")

    pivot_df = agg_df.pivot_table(
        index=["codice_istat", "nome_comune"],
        columns="pilastro",
        values="count",
        fill_value=0
    ).reset_index()

    pivot_df.columns.name = None

    for pillar in PILLARS.keys():
        if pillar not in pivot_df.columns:
            pivot_df[pillar] = 0

    print(f"  → {len(pivot_df)} comuni con dati POI")
    return pivot_df

In [ ]:
print(full_df["codice_istat"].dtype)

NameError: name 'full_df' is not defined

In [ ]:
# ============================================================
# MAIN : non farlo correre: il main è stato spezzato in celle
# ============================================================

"""def main():
    #Funzione principale per l'esecuzione dello script.

    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print("RACCOLTA DATI OSM - INDICE DI ATTRATTIVITÀ COMUNALE")
    print("=" * 60)
    print()

    geojson_data = download_geojson(GEOJSON_URL, timeout=REQUEST_TIMEOUT)
    filtered_geojson = filter_region(geojson_data, REGION_NAME)

    bbox = calculate_bbox(filtered_geojson)

    comuni_path = output_path / "comuni_sardegna.geojson"
    with open(comuni_path, "w", encoding="utf-8") as f:
        json.dump(filtered_geojson, f, ensure_ascii=False, indent=2)
    print(f"  → Salvato: {comuni_path}")

    comuni_gdf = gpd.read_file(comuni_path)
    print(f"  → Caricati {len(comuni_gdf)} comuni in GeoDataFrame")

    all_points = []
    for pillar_name, pillar_tags in PILLARS.items():
        print(f"\nQuery per pilastro: {pillar_name}")
        query = build_overpass_query(bbox, pillar_tags)

        try:
            data = query_overpass(OVERPASS_ENDPOINTS, query, timeout=REQUEST_TIMEOUT)
            points = extract_points_from_overpass(data)
            all_points.extend(points)
            print(f"  → {len(points)} elementi estratti")

            time.sleep(5)  # Rate limiting per evitare blocchi Overpass

        except Exception as e:
            print(f"  → ERRORE: {e}")
            continue

    joined_gdf = spatial_join_points_to_municipalities(all_points, comuni_gdf)

    counts_df = aggregate_counts(joined_gdf)

    full_df = comuni_gdf[["com_istat_code", "name"]].copy()
    full_df.columns = ["codice_istat", "nome_comune"]
    full_df["codice_istat"] = full_df["codice_istat"].astype(int)

    result_df = full_df.merge(counts_df, on=["codice_istat", "nome_comune"], how="left")
    result_df = result_df.fillna(0)

    for pillar in PILLARS.keys():
        if pillar in result_df.columns:
            result_df[pillar] = result_df[pillar].astype(int)

    csv_path = output_path / "osm_per_comune.csv"
    result_df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"\n[OK] Salvato: {csv_path}")
    print(f"  → {len(result_df)} righe, {len(result_df.columns)} colonne")

    print("\n" + "=" * 60)
    print("RACCOLTA DATI COMPLETATA")
    print("=" * 60)
    print(f"\nFile generati:")
    print(f"  - {comuni_path}")
    print(f"  - {csv_path}")
    print(f"\nProssimo step: python 02_calcolo_indice.py")
"""

In [ ]:
"""
if __name__ == "__main__":
    main()
"""

Main - spezzato

In [22]:
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

In [23]:
print(output_path)

/home/davide/Scaricati/sardegna-overtourism-aida26-main/output_attrattivita


In [24]:
print("=" * 60)
print("RACCOLTA DATI OSM - INDICE DI ATTRATTIVITÀ COMUNALE")
print("=" * 60)
print()


RACCOLTA DATI OSM - INDICE DI ATTRATTIVITÀ COMUNALE



In [25]:
geojson_data = download_geojson(GEOJSON_URL, timeout=REQUEST_TIMEOUT)
filtered_geojson = filter_region(geojson_data, REGION_NAME)
bbox = calculate_bbox(filtered_geojson)

[1/6] Scaricamento confini comunali da https://raw.githubusercontent.com/openpolis/geojson-italy/master/geojson/limits_IT_municipalities.geojson...
[2/6] Filtraggio per regione: Sardegna...
  → 377 comuni trovati in Sardegna
  → Bounding box calcolata: (38.8591910621176, 8.133825271143182, 41.31332055942893, 9.828420166045603)


In [26]:
comuni_path = output_path / "comuni_sardegna.geojson"
with open(comuni_path, "w", encoding="utf-8") as f:
    json.dump(filtered_geojson, f, ensure_ascii=False, indent=2)
print(f"  → Salvato: {comuni_path}")


  → Salvato: /home/davide/Scaricati/sardegna-overtourism-aida26-main/output_attrattivita/comuni_sardegna.geojson


In [27]:
comuni_gdf = gpd.read_file(comuni_path)
print(f"  → Caricati {len(comuni_gdf)} comuni in GeoDataFrame")


  → Caricati 377 comuni in GeoDataFrame


In [ ]:
# prova visualizzazione dati comuni
comuni_gdf.head(5)

,name,minint_elettorale,minint_finloc,op_id,prov_name,prov_istat_code,prov_istat_code_num,prov_acr,reg_name,reg_istat_code,reg_istat_code_num,opdm_id,com_catasto_code,com_istat_code,com_istat_code_num,geometry
0,Aggius,4200730010,5200730010,8124,Sassari,090,90,SS,Sardegna,20,20,7578,A069,090001,90001,"MULTIPOLYGON (((9.05067 41.06541, 9.05687 41.0..."
1,Alà dei Sardi,4200730020,5200730020,8126,Sassari,090,90,SS,Sardegna,20,20,7579,A115,090002,90002,"MULTIPOLYGON (((9.38623 40.74342, 9.40015 40.7..."
2,Alghero,4200730030,5200730030,7330,Sassari,090,90,SS,Sardegna,20,20,7580,A192,090003,90003,"MULTIPOLYGON (((8.33727 40.49933, 8.33697 40.4..."
3,Anela,4200730040,5200730040,7331,Sassari,090,90,SS,Sardegna,20,20,7581,A287,090004,90004,"MULTIPOLYGON (((8.99471 40.49346, 8.99711 40.4..."
4,Ardara,4200730050,5200730050,7332,Sassari,090,90,SS,Sardegna,20,20,7582,A379,090005,90005,"MULTIPOLYGON (((8.83475 40.66291, 8.84965 40.6..."


In [29]:
PILLARS.keys()

dict_keys(['turismo', 'natura', 'servizi', 'ristorazione', 'infrastrutture'])

In [30]:
all_points = []

In [31]:
#prova visualizzazione dei pilastri
for nome, tag in PILLARS.items():
    print(nome)

turismo
natura
servizi
ristorazione
infrastrutture


In [ ]:
#prova: all_points è una lista vuota
all_points

[]

In [51]:
pilastro = "turismo"
pillar_tags = PILLARS[pilastro]
query = build_overpass_query(bbox, pillar_tags)
try:
    data = query_overpass(OVERPASS_ENDPOINTS, query, timeout=REQUEST_TIMEOUT)
    points = extract_points_from_overpass(data)
    all_points.extend(points)
    print(f"  → {len(points)} elementi estratti")
    time.sleep(5)  # Rate limiting per evitare blocchi Overpass

except Exception as e:
    print(f"  → ERRORE: {e}")


  → Esecuzione query Overpass su https://overpass-api.de/api/interpreter (tentativo 1)...
  → 9764 elementi estratti


In [52]:
pilastro = "natura"
pillar_tags = PILLARS[pilastro]
query = build_overpass_query(bbox, pillar_tags)
try:
    data = query_overpass(OVERPASS_ENDPOINTS, query, timeout=REQUEST_TIMEOUT)
    points = extract_points_from_overpass(data)
    all_points.extend(points)
    print(f"  → {len(points)} elementi estratti")
    time.sleep(5)  # Rate limiting per evitare blocchi Overpass

except Exception as e:
    print(f"  → ERRORE: {e}")

  → Esecuzione query Overpass su https://overpass-api.de/api/interpreter (tentativo 1)...
  → 18881 elementi estratti


In [54]:
pilastro = "servizi"
pillar_tags = PILLARS[pilastro]
query = build_overpass_query(bbox, pillar_tags)
try:
    data = query_overpass(OVERPASS_ENDPOINTS, query, timeout=REQUEST_TIMEOUT)
    points = extract_points_from_overpass(data)
    all_points.extend(points)
    print(f"  → {len(points)} elementi estratti")
    time.sleep(5)  # Rate limiting per evitare blocchi Overpass

except Exception as e:
    print(f"  → ERRORE: {e}")

  → Esecuzione query Overpass su https://overpass-api.de/api/interpreter (tentativo 1)...
  → 2884 elementi estratti


In [61]:
pilastro = "ristorazione"
pillar_tags = PILLARS[pilastro]
query = build_overpass_query(bbox, pillar_tags)
try:
    data = query_overpass(OVERPASS_ENDPOINTS, query, timeout=REQUEST_TIMEOUT)
    points = extract_points_from_overpass(data)
    all_points.extend(points)
    print(f"  → {len(points)} elementi estratti")
    time.sleep(5)  # Rate limiting per evitare blocchi Overpass

except Exception as e:
    print(f"  → ERRORE: {e}")

  → Esecuzione query Overpass su https://overpass-api.de/api/interpreter (tentativo 1)...
  → 5965 elementi estratti


In [56]:
pilastro = "infrastrutture"
pillar_tags = PILLARS[pilastro]
query = build_overpass_query(bbox, pillar_tags)
try:
    data = query_overpass(OVERPASS_ENDPOINTS, query, timeout=REQUEST_TIMEOUT)
    points = extract_points_from_overpass(data)
    all_points.extend(points)
    print(f"  → {len(points)} elementi estratti")
    time.sleep(5)  # Rate limiting per evitare blocchi Overpass

except Exception as e:
    print(f"  → ERRORE: {e}")

  → Esecuzione query Overpass su https://overpass-api.de/api/interpreter (tentativo 1)...
  → 2305 elementi estratti


In [1]:
all_points

NameError: name 'all_points' is not defined

In [68]:
joined_gdf = spatial_join_points_to_municipalities(all_points, comuni_gdf)

[4/6] Costruzione GeoDataFrame dei punti POI...
  → 39799 punti creati
[5/6] Esecuzione spatial join...
  → 39799 punti associati a comuni


In [69]:
counts_df = aggregate_counts(joined_gdf)

[6/6] Aggregazione conteggi per comune...
  → 377 comuni con dati POI


In [70]:
full_df = comuni_gdf[["com_istat_code", "name"]].copy()
full_df.columns = ["codice_istat", "nome_comune"]
full_df["codice_istat"] = full_df["codice_istat"].astype(int)

In [80]:
full_df["codice_istat"]

0       90001
1       90002
2       90003
3       90004
4       90005
        ...  
372    111103
373    111104
374    111105
375    111106
376    111107
Name: codice_istat, Length: 377, dtype: int64

In [88]:
counts_df = counts_df.rename(columns={"codice_istat":"codice_istat_str"})

In [92]:
counts_df["codice_istat"] = counts_df["codice_istat_str"].astype(int)

In [93]:
counts_df

,codice_istat_str,nome_comune,infrastrutture,natura,ristorazione,servizi,turismo,codice_istat
0,090001,Aggius,0.0,19.0,14.0,6.0,12.0,90001
1,090002,Alà dei Sardi,2.0,62.0,16.0,5.0,40.0,90002
2,090003,Alghero,241.0,283.0,174.0,59.0,250.0,90003
3,090004,Anela,1.0,186.0,1.0,3.0,6.0,90004
4,090005,Ardara,2.0,14.0,3.0,4.0,8.0,90005
...,...,...,...,...,...,...,...,...
372,111103,Villaputzu,5.0,101.0,18.0,6.0,53.0,111103
373,111104,Villasalto,1.0,46.0,1.0,4.0,10.0,111104
374,111105,Villasimius,4.0,124.0,68.0,7.0,65.0,111105
375,111106,Villasor,7.0,252.0,6.0,6.0,19.0,111106


In [94]:
result_df = full_df.merge(counts_df, on=["codice_istat", "nome_comune"], how="left")

In [95]:
result_df = result_df.fillna(0)

In [96]:
for pillar in PILLARS.keys():
    if pillar in result_df.columns:
        result_df[pillar] = result_df[pillar].astype(int)

In [97]:
csv_path = output_path / "osm_per_comune.csv"

In [98]:
result_df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"\n[OK] Salvato: {csv_path}")
print(f"  → {len(result_df)} righe, {len(result_df.columns)} colonne")


[OK] Salvato: /home/davide/Scaricati/sardegna-overtourism-aida26-main/output_attrattivita/osm_per_comune.csv
  → 377 righe, 8 colonne


In [ ]:
print("\n" + "=" * 60)
print("RACCOLTA DATI COMPLETATA")
print("=" * 60)
print(f"\nFile generati:")
print(f"  - {comuni_path}")
print(f"  - {csv_path}")
print(f"\nProssimo step: python 02_calcolo_indice.py")
